## Problem Set 3: Text Classification Analysis
We will use the provided `archive (1)` dataset (training/validation/test) to perform end-to-end text classification: preprocessing, vectorization (TF-IDF), training three classifiers (MultinomialNB, LogisticRegression, LinearSVC), and evaluation (accuracy, precision/recall/F1, confusion matrices).

### Step 1 — Dataset Selection & Loading
We will use the supplied dataset inside the `archive (1)` folder. The dataset contains `training.csv`, `validation.csv`, and `test.csv` with two columns: `text` and `label`. Load and inspect the files to understand the data structure and label distribution.

In [ ]:
# install required packages in notebook environment
%pip install matplotlib seaborn scikit-learn pandas nltk

import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import make_pipeline

# Load dataset files from archive (1)
data_dir = 'archive (1)'
train = pd.read_csv(os.path.join(data_dir, 'training.csv'))
val = pd.read_csv(os.path.join(data_dir, 'validation.csv'))
test = pd.read_csv(os.path.join(data_dir, 'test.csv'))
print('Train size:', len(train), 'Validation size:', len(val), 'Test size:', len(test))
display(train.head())
print('Label distribution (train):')
print(train['label'].value_counts().sort_index())

### Step 2 — Data Preprocessing

**Goal:** Clean and prepare text data for machine learning by removing noise and standardizing format.

**Process:**
- **Lowercasing:** Convert all text to lowercase for uniformity
- **Remove URLs:** Strip out hyperlinks which don't contribute to text classification
- **Remove Special Characters:** Keep only letters and spaces
- **Remove Extra Whitespace:** Normalize spacing
- **Stop Word Removal:** Exclude common words (and, the, is, etc.) that don't carry meaning
- **Tokenization:** Break text into individual words

This preprocessing step ensures that our text data is clean and normalized before feeding it into machine learning algorithms.

In [ ]:
# Preprocessing: simple cleaning and stop word handling
def clean_text(s):
    if not isinstance(s, str):
        return ''
    s = s.lower()
    s = re.sub(r'http\S+', ' ', s)
    s = re.sub(r'[^a-z\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s
# load optional Tagalog stop words (if present) and combine with sklearn English stop words
tagalog_path = os.path.join('Text Classification', 'tagalog_stop_words.txt')
tagalog_sw = set()
if os.path.exists(tagalog_path):
    with open(tagalog_path, 'r', encoding='utf-8') as f:
        tagalog_sw = set([w.strip() for w in f if w.strip()])
stop_words = set(ENGLISH_STOP_WORDS).union(tagalog_sw)
# Combine train + val for training
df_train = pd.concat([train, val], ignore_index=True)
df_train['text_clean'] = df_train['text'].apply(clean_text)
df_test = test.copy()
df_test['text_clean'] = df_test['text'].apply(clean_text)
print('Sample cleaned text:')
display(df_train[['text', 'text_clean']].head())

### Step 3 — Vectorization & Model Training

**Goal:** Convert text into numerical features and train machine learning classifiers.

#### TF-IDF (Term Frequency-Inverse Document Frequency)
TF-IDF is a technique to measure the importance of words in documents:

- **Term Frequency (TF):** How often a word appears in a document
- **Inverse Document Frequency (IDF):** How unique/rare a word is across all documents  
- **Combined TF-IDF:** High scores for words that appear frequently in some documents but rarely across the collection

**Why Use TF-IDF?**
It emphasizes unique, meaningful words while downplaying common words. For example, in a news classification dataset, specialized words like "legislative" or "sports" are more important than generic words like "the" or "is".

**Models Trained:**
1. **MultinomialNB** - Naive Bayes classifier (good for text, fast)
2. **LogisticRegression** - Linear classifier with probabilistic output
3. **LinearSVC** - Support Vector Machine for linear classification

In [ ]:
# Step 3: Vectorize and train models (Enter code here)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

# ensure stop_words is a list when passing to vectorizer
try:
    sw_list = list(stop_words)
except NameError:
    sw_list = None

vec = TfidfVectorizer(min_df=3, max_df=0.9, ngram_range=(1,2), stop_words=sw_list)
X_train = vec.fit_transform(df_train['text_clean'].fillna(''))
X_test = vec.transform(df_test['text_clean'].fillna(''))
y_train = df_train['label'].astype(int)
y_test = df_test['label'].astype(int)
print('Vocab size:', len(vec.get_feature_names_out()))
models = [('MultinomialNB', MultinomialNB()), ('LogisticRegression', LogisticRegression(max_iter=2000)), ('LinearSVC', LinearSVC(max_iter=20000))]
trained = {}
for name, model in models:
    print('Training', name)
    model.fit(X_train, y_train)
    trained[name] = model
print('Training complete')

### Step 4 — Model Evaluation and three classification algorithms

**Goal:** Assess classifier performance using multiple metrics.

**Metrics Used:**

- **Accuracy:** Overall correctness (correct predictions / total predictions)
- **Precision:** Of predicted positive cases, how many are actually correct
- **Recall:** Of actual positive cases, how many did we correctly identify
- **F1-Score:** Harmonic mean of precision and recall (balances both)
- **Confusion Matrix:** Visualization showing true positives, false positives, false negatives, true negatives

**Interpretation:**
- High accuracy doesn't always mean good model (especially with imbalanced data)
- Precision vs Recall trade-off: optimize based on problem requirements
- Confusion matrix reveals which classes are commonly confused

**Classification algorithms**
- Multinomial NB
- Logistic Regression
- Support Vector Machine for Classification of linear

In [ ]:
# Train three classifiers and evaluate
models = [
    ('MultinomialNB', MultinomialNB()),
    ('LogisticRegression', LogisticRegression(max_iter=2000)),
    ('LinearSupportVectorMachine', LinearSVC(max_iter=20000))
]
labels = sorted(list(set(y_train.unique()).union(set(y_test.unique()))))
results = {}
for name, model in models:
    print('Training', name)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, digits=4)
    cm = confusion_matrix(y_test, y_pred, labels=labels)
    results[name] = {'accuracy': acc, 'report': report, 'cm': cm}
    print(f'--- {name} ---')
    print('Accuracy:', acc)
    print(report)
    # plot confusion matrix inline
    plt.figure(figsize=(6,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.title(f'Confusion matrix: {name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

**Summary & Key Findings**
- The notebook trained three classifiers (MultinomialNB, LogisticRegression, LinearSVC) and evaluated each with:
  - **Accuracy scores** for overall performance
  - **Classification reports** showing precision, recall, and F1-scores for each class
  - **Confusion matrices** revealing per-class performance and misclassification patterns


### Step 5 — Model Comparison and Results Summary

**Compare the three models based on:**

1. **Accuracy:** Which model has the highest overall correctness?
2. **Precision/Recall Balance:** Does one model excel at recall while another has better precision?
3. **F1-Score:** Best harmonic balance between precision and recall?
4. **Confusion Matrix Analysis:** 
   - Which classes are confused with each other?
   - Does one model perform better on specific classes?
   - Are there systematic errors (e.g., always misclassifying one category)?



In [ ]:
# Step 5: Model Comparison and Analysis
import pandas as pd

# 1. Accuracy Comparison
print("=" * 60)
print("1. ACCURACY COMPARISON")
print("=" * 60)
accuracy_data = [(name, results[name]['accuracy']) for name in results.keys()]
accuracy_df = pd.DataFrame(accuracy_data, columns=['Model', 'Accuracy'])
accuracy_df = accuracy_df.sort_values('Accuracy', ascending=False)
print(accuracy_df.to_string(index=False))
best_accuracy = accuracy_df.iloc[0]
print(f"\n✓ Best Accuracy: {best_accuracy['Model']} ({best_accuracy['Accuracy']:.4f})")

# 2. Precision, Recall, F1-Score Analysis
print("\n" + "=" * 60)
print("2. PRECISION, RECALL & F1-SCORE SUMMARY")
print("=" * 60)
for name in results.keys():
    print(f"\n{name}:")
    print(results[name]['report'])

# 3. Confusion Matrix Analysis
print("\n" + "=" * 60)
print("3. CONFUSION MATRIX ANALYSIS")
print("=" * 60)
for name in results.keys():
    cm = results[name]['cm']
    correct = np.trace(cm)  # diagonal = correct predictions
    total = np.sum(cm)
    print(f"\n{name}:")
    print(f"  Correct predictions: {correct}/{total}")
    print(f"  Per-class accuracy: {[f'{cm[i,i]}/({np.sum(cm[i,:])} total)' for i in range(len(labels))]}")

# 4. Overall Summary and Recommendation
print("\n" + "=" * 60)
print("4. OVERALL RECOMMENDATION")
print("=" * 60)
best_model = accuracy_df.iloc[0]['Model']
print(f"\n✓ Best Overall Model: {best_model}")
print(f"  - Highest accuracy: {accuracy_df.iloc[0]['Accuracy']:.4f}")
print(f"  - Second best: {accuracy_df.iloc[1]['Model']} ({accuracy_df.iloc[1]['Accuracy']:.4f})")
print(f"  - Third: {accuracy_df.iloc[2]['Model']} ({accuracy_df.iloc[2]['Accuracy']:.4f})")

print("\nKey Findings:")
print("  • Review classification reports for precision/recall trade-offs")
print("  • Confusion matrices show which classes are most often confused")
print("  • Consider domain requirements when choosing between models")
print("  • High F1-score indicates good balance between precision and recall")